# Masked autoencoders for self-supervised learning

In this notebook, we will briefly discuss the **masked autoencoder** (MAE) [[He et al., 2021](https://arxiv.org/abs/2111.06377)]. The MAE is a scheme for self-supervised representation learning that is especially well-suited to the **vision transformer** (ViT) [[Dosovskiy et al., 2020](https://arxiv.org/abs/2010.11929)]. A masking strategy is used to automatically generate a supervisory signal from unlabeled images. This approach enables learning useful representations without the need for large amounts of labeled data, making it particularly valuable in the common situation in which labeled data is scarce and expensive to obtain. Nowadays, the MAE is a widely used technique for pretraining vision foundation models.

## Self-supervised learning

In **representation learning**, one tries to encode the raw data into a more meaningful representation. This encoding may simply be more compact, or it could be more relevant for a certain downstream task. While there are a variety of supervised and unsupervised approaches that may yield such useful features, **self-supervised learning** (SSL) subsumes strategies where a training signal is extracted automatically from unlabeled data [[Ozbulak et al., 2023](https://arxiv.org/abs/2305.13689); [Balestriero et al., 2023](https://arxiv.org/abs/2304.12210); [Gui et al., 2024](https://arxiv.org/abs/2301.05712); [Uelwer et al., 2025](https://doi.org/10.1007/s10994-024-06708-7)]. While SSL is generally unsupervised in the sense that it does not require pre-existing labels, many methods could also be seen as supervised in that they train on auto-generated input-target pairs.

The **denoising autoencoder** (DAE) [[Vincent et al., 2008](https://doi.org/10.1145/1390156.1390294); [Vincent et al., 2010](https://www.jmlr.org/papers/v11/vincent10a.html)] is a prototypical SSL approach. Here, an autoencoder is trained to reconstruct an image from a corrupted version of it, i.e. the reconstruction error between the original and the reconstructed images is minimized. This denoising task encourages the latent representation to capture the underlying structure of the data.

Other so-called **pretext tasks** for visual representation learning can be similarly seen as image-space information restoration problems. These include the recolorization of grayscale images [[Larsson et al., 2017](https://arxiv.org/abs/1703.04044)], jigsaw puzzle solving [[Noroozi and Favaro, 2016](https://arxiv.org/abs/1603.09246)] and, most notably in the context of this notebook, image inpainting [[Pathak et al., 2016](https://arxiv.org/abs/1604.07379)]. The latter idea has evolved into a vast set of methods called **masked image modeling** (MIM) [[Hondru et al., 2025](https://doi.org/10.1007/s11263-025-02524-1)] that spans restoration as well as other SSL approaches.

Autoassociative pretext tasks are not the only way to formulate SSL. As just mentioned above, other approaches are based on **contrastive learning**. Here, the embeddings of differently augmented views of the same image are pulled closer together, while the embeddings of genuinely different images are pushed further apart [[Misra and van der Maaten, 2019](https://arxiv.org/abs/1912.01991); [Chen et al., 2020](https://arxiv.org/abs/2002.05709)].

Another line of work is based on a **self-distillation** scheme in conjunction with a **student-teacher** architecture [[Grill et al., 2020](https://arxiv.org/abs/2006.07733); [Caron et al., 2021](https://arxiv.org/abs/2104.14294); [Zhou et al., 2022](https://arxiv.org/abs/2111.07832)]. During training, student and teacher networks are shown differently transformed versions of the same image. The student is trained to output a representation that matches the output of the teacher using gradient descent, while the teacher is more slowly updated as a moving average of the student.

## MAE pretraining

The **masked autoencoder** (MAE) is an influential reconstruction-based SSL scheme that was proposed by [[He et al., 2021](https://arxiv.org/abs/2111.06377)]. It employs a patch-wise masking scheme together with a ViT-based encoder-decoder architecture that is trained to restore the original input. This masking-then-restoring approach gives rise to a meaningful latent representation.

An image is first decomposed into regular non-overlapping patches. These are exactly the patches from the ViT. Patches are then randomly removed from the input, and only the non-masked patches are fed to the ViT encoder. A positional embedding is used as usual. Following this, the ViT decoder reads the encodings of the visible patches and a shared learnable mask token for each invisible patch. A positional embedding is added to all tokens, including the mask tokens. The original MAE paper illustrated the architecture as follows:

<img src="https://upload.wikimedia.org/wikipedia/commons/6/6c/Masked_Autoencoder.svg" alt="The original MAE architecture is shown" title="MAE architecture from He et al., 2021" height="300" style="background-color: white;">

A surprisingly high percentage of image patches (e.g. 75%) is removed such that only a fraction enters the encoder. The full set of patches is only handled by a comparably lightweight decoder. These design choices reduce the computational expense of training MAEs. After training, the MAE encoder outputs the latent image representation. The auxiliary decoder can be simply removed.